# Practice 58 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Staff performance reviews

Given `reviews` below — `perf` and `engage` scores recorded in 2023 and 2024 for six employees across two departments:

- Reshape into long format. Each row should represent one `(staff_id, year)` pair, with `perf` and `engage` as separate columns.
- Which department — Ops or Tech — improved more from 2023 to 2024? (mean improvement across both scores)
- Use `np.corrcoef` to check whether `perf` and `engage` are correlated across all rows. Does higher engagement track with higher performance?

In [22]:
reviews = pd.DataFrame({
    'staff_id':     [101, 102, 103, 104, 105, 106],
    'dept':         ['Ops', 'Tech', 'Tech', 'Ops', 'Tech', 'Ops'],
    'perf_2023':    [72, 88, 95, 65, 81, 70],
    'perf_2024':    [78, 85, 97, 74, 86, 79],
    'engage_2023':  [60, 75, 85, 55, 70, 58],
    'engage_2024':  [68, 78, 90, 62, 76, 66],
})

# Your code here

reviews_long = pd.wide_to_long(
    reviews, 
    stubnames=['perf','engage'],
    i = ['staff_id','dept'],
    j = 'year',
    sep = '_',
    suffix= r'\d+'
)

r3 = reviews_long.xs(2023, level='year')
r4 = reviews_long.xs(2024, level='year')
r3_mean = r3.groupby(level='dept')[['perf','engage']].mean()
r4_mean = r4.groupby(level='dept')[['perf','engage']].mean()
print((r4_mean - r3_mean).sum(1).idxmax(), ' improved more')

np.corrcoef(reviews_long['perf'], reviews_long['engage'])[0,1]
print('highly correlated')

Ops  improved more
highly correlated


---

## Level 2 — Restaurant tips

Use the seaborn `tips` dataset. No hints.

1. Build a crosstab of `smoker` vs `day`, normalized by **column**. On which day is the smoker proportion highest?
2. Build a crosstab of `sex` vs `time`, normalized by **row**. Among females, what fraction dine at lunch?
3. Add a `tip_pct` column (`tip / total_bill`). For each `day`, use `np.percentile` to find the 25th and 75th percentiles of `tip_pct`. Which day has the widest spread (largest IQR)?

In [46]:
tips = sns.load_dataset('tips')

# Your code here

p = pd.crosstab(tips['smoker'], tips['day'], normalize='columns')
print(p.loc['Yes'].idxmax(), 'is the smoker proportion highest')

ps = pd.crosstab(tips['sex'],tips['time'], normalize= 'index')
print(ps.loc['Female','Lunch'], 'dine at lunch')

tips['tip_pct'] = tips['tip']/tips['total_bill']
tg = tips.groupby('day', observed=True)['tip_pct'].apply(lambda x: np.percentile(x, [25,75]))
print(tg)
print(tg.index[np.argmax([tg[i][1]-tg[i][0] for i in tg.index])], 'has the widest IQR')

Fri is the smoker proportion highest
0.40229885057471265 dine at lunch
day
Thur    [0.13820958384128318, 0.19268674587823525]
Fri     [0.13373871366324785, 0.19663728652788104]
Sat     [0.12386329402050812, 0.18827081506393373]
Sun     [0.11998207788269949, 0.18788908169367569]
Name: tip_pct, dtype: object
Sun has the widest IQR


In [75]:
(tg.apply(lambda x: x[1] - x[0])).idxmax()

'Sun'

---

## Level 3 — Fuel economy

Use `sns.load_dataset('mpg')`. No hints.

1. Create an `era` column: `'early'` for `model_year < 75`, `'late'` otherwise. Build a crosstab of `era` vs `origin`, normalized by row. How did the share of each origin shift between eras?
2. Drop rows where `horsepower` is null. Use `np.corrcoef` to compute the correlation between `mpg` and each of: `horsepower`, `weight`, `displacement`. Which predictor is most strongly (negatively) correlated with fuel efficiency?
3. Filter to `origin == 'usa'`. Use `np.argsort` to rank these cars by `mpg` descending. What are the top-3 most fuel-efficient USA car names?

In [73]:
mpg = sns.load_dataset('mpg')

# Your code here

mpg['era'] = np.where(mpg['model_year']<75, 'early','late')

pp = pd.crosstab(mpg['era'], mpg['origin'], normalize='index')

print(pp)
print(pp.loc['late']-pp.loc['early'])

mpg = mpg.dropna(subset=['horsepower'])
ci =  ['horsepower','weight','displacement']
cors = [np.corrcoef(mpg['mpg'], mpg[c])[0,1] for c in ci]
print(cors)
print(ci[np.argmin(cors)]," is most negatively correlated")

usa = mpg.loc[mpg['origin'] == 'usa']
usa_sorted = usa.iloc[np.argsort(-usa['mpg'])]
usa_sorted.iloc[:3]


origin    europe     japan       usa
era                                 
early   0.177632  0.138158  0.684211
late    0.174797  0.235772  0.589431
origin
europe   -0.002835
japan     0.097614
usa      -0.094780
dtype: float64
[np.float64(-0.7784267838977756), np.float64(-0.8322442148315753), np.float64(-0.8051269467104578)]
weight  is most negatively correlated


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name,era
344,39.0,4,86.0,64.0,1875,16.4,81,usa,plymouth champ,late
387,38.0,6,262.0,85.0,3015,17.0,82,usa,oldsmobile cutlass ciera (diesel),late
378,38.0,4,105.0,63.0,2125,14.7,82,usa,plymouth horizon miser,late
